# Canadian Pipeline Network — Interactive Map with Local AI Chat

Combines several Canadian pipeline datasets into one clean table and shows them on an interactive **Mapbox** map. A chat sidebar lets you filter the map in plain English (e.g. *"show oil pipelines in Alberta"*), powered by an open **Qwen** model that runs locally on the Colab GPU.

**Before you run**
1. Set the runtime to a GPU: **Runtime → Change runtime type → T4 GPU**.
2. Add Colab Secrets (key icon in the sidebar, notebook access ON): `MAPBOX_ACCESS_TOKEN` (required) and `HF_TOKEN` (optional, speeds up model download).
3. Put your datasets in Google Drive and update the paths in section 1.4.

**Sections**
1. Setup (install, imports, Drive, config, tokens)
2. Data cleaning
3. Map builder
4. Local AI model
5. Run (clean data → launch map)

See `README.md` for full details.

## 1. Setup

### 1.1 Install dependencies
Run once per fresh Colab runtime. `torch` is already included in Colab.

In [ ]:
!pip -q install geopandas shapely pandas requests openpyxl transformers accelerate

### 1.2 Imports
All imports used by the notebook, in one place.

In [ ]:
import html
import json
import os
import re
import threading
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path

import geopandas as gpd
import pandas as pd
import requests
from google.colab import drive, output, userdata
from shapely import wkt
from shapely.geometry import MultiLineString, box, mapping, shape
from shapely.ops import unary_union
from shapely.prepared import prep

### 1.3 Connect Google Drive
Used to read the source datasets. Skip this cell if you upload the files to `/content` instead.

In [ ]:
drive.mount("/content/drive")

### 1.4 Configure data files
Edit `DATA_FOLDER` and the file names to match your Drive. Supported formats: CSV, TXT, Excel, GeoJSON, JSON. The dictionary keys are dataset labels stored in the `source_dataset` column.

In [ ]:
# Folder in Google Drive that contains the pipeline datasets (CHANGE THIS)
DATA_FOLDER = Path("/content/drive/MyDrive/Pipelines")

file_paths = {
    "Pipeline_Canada": DATA_FOLDER / "Pipeline_Canada.csv",
    "Pipeline_Canada2": DATA_FOLDER / "Pipeline_Canada2.geojson",
    "Pipeline_Segments_BC": DATA_FOLDER / "Pipeline_Segments_BC.csv",
    "Pipeline_AB": DATA_FOLDER / "Pipeline_AB.csv",
    "Pipelines_NB": DATA_FOLDER / "Pipeline_NB.csv",
}

# Output folder for the cleaned CSV
BASE_DIR = Path("/content")
DATA_DIR = BASE_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Fail early if any input file is missing
missing_files = [str(path) for path in file_paths.values() if not Path(path).is_file()]
if missing_files:
    raise FileNotFoundError(
        "These dataset files were not found. Update DATA_FOLDER/file_paths:\n"
        + "\n".join(missing_files)
    )

print(f"Found {len(file_paths)} input datasets.")
for dataset_name, path in file_paths.items():
    print(f"  {dataset_name}: {path}")

### 1.5 Load access tokens
Tokens are read from **Colab Secrets** first, then from existing **environment variables**, and exported as environment variables.

| Name | Required | Purpose |
|---|---|---|
| `MAPBOX_ACCESS_TOKEN` (or `MAPBOX_TOKEN`) | Yes | Draws the Mapbox basemap |
| `HF_TOKEN` | No | Authenticated Hugging Face model download (faster, fewer rate limits) |

Use a Mapbox **public** token (`pk.…`): it is embedded in the generated map page.

In [ ]:
def get_secret(*names):
    """Return the first value found in Colab Secrets, then in environment variables."""
    for name in names:
        try:
            value = userdata.get(name)
            if value:
                return value
        except Exception:
            pass  # secret not defined or notebook access not enabled

    for name in names:
        value = os.environ.get(name)
        if value:
            return value

    return None


# ---------- Mapbox (required) ----------
MAPBOX_TOKEN = get_secret("MAPBOX_ACCESS_TOKEN", "MAPBOX_TOKEN")
if not MAPBOX_TOKEN:
    raise ValueError(
        "Mapbox token not found. Add MAPBOX_ACCESS_TOKEN to Colab Secrets "
        "(with notebook access enabled) or set it as an environment variable."
    )
os.environ["MAPBOX_ACCESS_TOKEN"] = MAPBOX_TOKEN
print("Mapbox token loaded.")

# ---------- Hugging Face (optional) ----------
# huggingface_hub reads the HF_TOKEN environment variable automatically.
HF_TOKEN = get_secret("HF_TOKEN")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("Hugging Face token loaded.")
else:
    print("No HF_TOKEN found. Continuing without it (the open models still download fine).")

## 2. Data cleaning
Turns the differently-structured source files into **one standard table** (`STANDARD_COLUMNS`), keeps the original attributes as JSON, stores geometry as WKT, and removes empty rows and duplicates.

### 2.1 Standard schema and column aliases
`STANDARD_COLUMNS` is the output layout. `ALIASES` lists the possible source column names for each standard field (matched after normalization to lowercase snake_case, first match wins).

In [ ]:
# Final column layout of the cleaned dataset
STANDARD_COLUMNS = [
    "pipeline_name",
    "company",
    "parent_company",
    "province",
    "location_description",
    "from_location",
    "to_location",
    "commodity",
    "pipeline_length_km",
    "capacity",
    "status",
    "start_date",
    "end_date",
    "segment_number",
    "licence_number",
    "project_number",
    "authority_type",
    "data_source",
    "latitude",
    "longitude",
    "geometry",
    "source_dataset",
    "source_record_id",
    "source_fields",
]


# Possible source column names for each standard field
ALIASES = {
    "pipeline_name": [
        "pipeline name", "name", "name_en", "name_fr"
    ],
    "company": [
        "cer-regulated company", "company_name", "proponent", "company"
    ],
    "parent_company": [
        "parent company"
    ],
    "province": [
        "province", "province_name"
    ],
    "location_description": [
        "location (description)", "location", "location_description"
    ],
    "from_location": [
        "from_location", "from location"
    ],
    "to_location": [
        "to_location", "to location"
    ],
    "commodity": [
        "transported commodity", "commodity", "substance 1", "line_type_desc"
    ],
    "pipeline_length_km": [
        "cer-regulated pipeline length (km)", "pipeline_length_km"
    ],
    "capacity": [
        "pipeline capacity", "capacity"
    ],
    "status": [
        "status", "segment_status"
    ],
    "start_date": [
        "commenced operations", "start_date", "approval_date", "approval date"
    ],
    "end_date": [
        "end_date", "retirement_date"
    ],
    "segment_number": [
        "segment_number"
    ],
    "licence_number": [
        "licence_number", "license_number"
    ],
    "project_number": [
        "project_number"
    ],
    "authority_type": [
        "authority_type"
    ],
    "data_source": [
        "data_source"
    ],
}

### 2.2 File-reading helpers
Normalize column names and read CSV / TXT / Excel / GeoJSON / JSON files.

In [ ]:
def normalize_column(name):
    """Convert column names into consistent lowercase snake_case."""
    name = str(name).strip().lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    return name.strip("_")


def read_csv_safely(path):
    """Read comma-separated CSV files using pandas' fast C parser."""
    last_error = None

    # Try common encodings in turn; government datasets are often cp1252.
    for encoding in ("utf-8", "cp1252", "latin1"):
        try:
            return pd.read_csv(
                path,
                sep=",",
                encoding=encoding,
                engine="c",
                low_memory=False,
                on_bad_lines="skip",  # drop malformed rows instead of failing
            )
        except UnicodeDecodeError as error:
            last_error = error

    raise ValueError(
        f"Could not read {path}. Last encoding error: {last_error}"
    )


def read_file(path):
    """Read CSV, TXT, Excel, GeoJSON, and JSON files."""
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix in (".geojson", ".json"):
        return gpd.read_file(path)

    if suffix in (".xlsx", ".xls"):
        return pd.read_excel(path)

    if suffix in (".csv", ".txt"):
        return read_csv_safely(path)

    raise ValueError(f"Unsupported file format: {path.suffix}")

### 2.3 Geometry helpers
Parse WKT or GeoJSON geometry safely (NaN, empty and invalid values become `None`) and compute representative coordinates.

In [ ]:
def parse_wkt_geometry(value):
    """Parse a WKT string, returning None when it is missing or invalid."""
    if not isinstance(value, str) or not value.strip():
        return None

    try:
        return wkt.loads(value)
    except Exception:
        return None


def is_valid_geometry(geom):
    """
    True only for a non-missing, non-empty Shapely geometry.

    This explicitly rejects NaN values (which pandas represents as floats),
    strings, and other non-geometry objects.
    """
    return (
        geom is not None
        and hasattr(geom, "geom_type")
        and hasattr(geom, "is_empty")
        and not geom.is_empty
    )


def normalize_geometry_value(value):
    """Return a Shapely geometry or None, regardless of input cell type."""
    if is_valid_geometry(value):
        return value

    if isinstance(value, str):
        parsed = parse_wkt_geometry(value)
        return parsed if is_valid_geometry(parsed) else None

    return None


def get_geometries(df):
    """Find GeoJSON geometry or a WKT geometry column and normalize values."""
    geometry_values = None

    if isinstance(df, gpd.GeoDataFrame):
        geometry_values = df.geometry

    else:
        for column in ("the_geom", "geom", "wkt", "geometry"):
            if column in df.columns:
                geometry_values = df[column]
                break

    if geometry_values is None:
        return pd.Series(None, index=df.index, dtype=object)

    # Always normalize each cell. This safely handles NaN floats, None,
    # WKT strings, and already-parsed Shapely geometries.
    return geometry_values.apply(normalize_geometry_value)


def representative_coordinate(geom, coordinate):
    """Return the latitude or longitude of a point guaranteed to lie on the geometry."""
    if not is_valid_geometry(geom):
        return None

    point = geom.representative_point()
    return point.y if coordinate == "latitude" else point.x

### 2.4 Record helpers
Pick a source record ID for de-duplication and keep the original attributes as a JSON string.

In [ ]:
def get_source_record_id(df):
    """Find the best available source record identifier."""
    id_fields = [
        "objectid",
        "pinfo_id",
        "licence_number",
        "project_number",
        "segment_number",
    ]

    result = pd.Series(None, index=df.index, dtype=object)

    for column in id_fields:
        if column in df.columns:
            result = result.fillna(df[column])

    return result


def serialize_source_fields(df):
    """
    Preserve original non-geometry attributes as JSON.
    Geometry is excluded to avoid duplicating large WKT values.
    """
    geometry_columns = {"geometry", "the_geom", "geom", "wkt"}
    source_columns = [
        column for column in df.columns
        if column not in geometry_columns
    ]

    records = df[source_columns].to_dict(orient="records")

    return [
        json.dumps(record, ensure_ascii=False, default=str)
        for record in records
    ]

### 2.5 Main cleaning function
`clean_pipeline_dataset()` loads every dataset, maps columns to the standard schema, combines everything, removes duplicates and writes `Canadian_Pipelines_Clean.csv` to `/content/data/`.

In [ ]:
def clean_pipeline_dataset(
    file_paths,
    output_csv="Canadian_Pipelines_Clean.csv",
):
    """
    Combine pipeline datasets into one standardized CSV.

    Parameters
    ----------
    file_paths : dict
        Dictionary mapping dataset names to file paths.
    output_csv : str or pathlib.Path
        Output filename. Saved inside DATA_DIR (/content/data) unless an
        absolute path is supplied.

    Returns
    -------
    pandas.DataFrame
        Combined cleaned pipeline dataset.
    """

    output_path = Path(output_csv)

    # Save relative output paths in the project's data folder.
    if not output_path.is_absolute():
        output_path = DATA_DIR / output_path

    output_path.parent.mkdir(parents=True, exist_ok=True)

    all_records = []

    # ---- Step 1: load and standardize each dataset ----
    for dataset_name, file_path in file_paths.items():
        print(f"\nReading {dataset_name}...")

        df = read_file(file_path)

        # Normalize source column names.
        df.columns = [normalize_column(column) for column in df.columns]

        print(f"Loaded {len(df):,} records.")

        cleaned = pd.DataFrame(index=df.index)

        # Map source columns to standardized columns.
        for standard_field, aliases in ALIASES.items():
            matching_column = next(
                (
                    normalize_column(alias)
                    for alias in aliases
                    if normalize_column(alias) in df.columns
                ),
                None,
            )

            if matching_column is not None:
                cleaned[standard_field] = df[matching_column]
            else:
                cleaned[standard_field] = None

        cleaned["source_dataset"] = dataset_name
        cleaned["source_record_id"] = get_source_record_id(df)

        # Parse and normalize geometries once.
        geometries = get_geometries(df)

        # Save geometry as WKT for CSV compatibility.
        cleaned["geometry"] = geometries.apply(
            lambda geom: geom.wkt if is_valid_geometry(geom) else None
        )

        # Representative-point coordinates (used for lat/lon columns).
        cleaned["latitude"] = geometries.apply(
            lambda geom: representative_coordinate(geom, "latitude")
        )
        cleaned["longitude"] = geometries.apply(
            lambda geom: representative_coordinate(geom, "longitude")
        )

        # Preserve original non-geometry attributes.
        cleaned["source_fields"] = serialize_source_fields(df)

        # Ensure all output columns exist and are in the same order.
        for column in STANDARD_COLUMNS:
            if column not in cleaned.columns:
                cleaned[column] = None

        cleaned = cleaned[STANDARD_COLUMNS]

        # Remove rows with no useful identifying information.
        useful_columns = [
            "pipeline_name",
            "company",
            "geometry",
            "location_description",
            "from_location",
            "licence_number",
            "project_number",
        ]

        cleaned = cleaned.dropna(how="all", subset=useful_columns)

        all_records.append(cleaned)

        print(
            f"Processed {len(cleaned):,} records from {dataset_name}."
        )

    if not all_records:
        raise ValueError("No datasets were supplied in file_paths.")

    # ---- Step 2: combine, de-duplicate and tidy ----
    # Combine datasets.
    print("\nCombining datasets...")
    combined = pd.concat(all_records, ignore_index=True)

    # Convert numeric fields without assuming unknown source length units.
    for column in ("pipeline_length_km", "latitude", "longitude"):
        combined[column] = pd.to_numeric(
            combined[column], errors="coerce"
        )

    # Remove duplicate records only when a source ID is available.
    has_id = combined["source_record_id"].notna()

    identified = combined.loc[has_id].drop_duplicates(
        subset=["source_dataset", "source_record_id"],
        keep="first",
    )
    unidentified = combined.loc[~has_id]

    combined = pd.concat(
        [identified, unidentified],
        ignore_index=True,
    )

    # Trim whitespace in text columns.
    for column in combined.select_dtypes(include="object").columns:
        combined[column] = combined[column].apply(
            lambda value: value.strip() if isinstance(value, str) else value
        )

    # ---- Step 3: export ----
    # utf-8-sig so the CSV opens cleanly in Excel.
    combined.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig",
    )

    print("\n----------------------------------")
    print("PIPELINE DATA CLEANING COMPLETE")
    print("----------------------------------")
    print(f"Total records: {len(combined):,}")
    print(f"Output file: {output_path.resolve()}")
    print("----------------------------------")

    return combined

## 3. Map builder
Clips pipelines to Canada, greys out everything outside the border, and produces the full interactive page (Mapbox GL JS + chat sidebar) as an HTML string.

### 3.1 Map constants

In [ ]:
# Natural Earth country polygons (Canada is extracted from this file)
CANADA_GEOJSON_URL = (
    "https://raw.githubusercontent.com/nvkelso/"
    "natural-earth-vector/master/geojson/"
    "ne_10m_admin_0_countries.geojson"
)

# Maximum area the user can pan to: [[west, south], [east, north]]
CANADA_BOUNDS = [
    [-150, 40],
    [-45, 85]
]

# Tolerance (degrees, ~2 km) so coastal / border pipelines are not
# trimmed just because the boundary file is slightly simplified.
CLIP_TOLERANCE_DEG = 0.02

### 3.2 Canada boundary and clipping
Download the boundary and build the function that removes pipeline parts outside Canada.

In [ ]:
def get_canada_boundary():
    """Download Natural Earth country polygons and return Canada as one Shapely geometry."""

    print("Downloading Canada boundary...")

    response = requests.get(CANADA_GEOJSON_URL, timeout=60)
    response.raise_for_status()

    world_data = response.json()

    canada_features = [
        feature
        for feature in world_data["features"]
        if feature["properties"].get("ADMIN") == "Canada"
    ]

    if not canada_features:
        raise ValueError("Could not find Canada in boundary dataset.")

    return unary_union([
        shape(feature["geometry"])
        for feature in canada_features
    ])


def _keep_lines(geom):
    """After clipping, keep only line parts (drops stray points)."""
    if geom is None or geom.is_empty:
        return None

    if geom.geom_type in ("LineString", "MultiLineString"):
        return geom

    if geom.geom_type == "GeometryCollection":
        lines = []
        for part in geom.geoms:
            if part.geom_type == "LineString":
                lines.append(part)
            elif part.geom_type == "MultiLineString":
                lines.extend(part.geoms)
        return MultiLineString(lines) if lines else None

    return geom


def build_canada_clipper(canada_geometry):
    """
    Return a clip(geom) function that keeps only the part of a pipeline inside Canada.

    clip(geom) returns (geometry_or_None, "kept" | "clipped" | "removed").
    The boundary is simplified and buffered slightly (CLIP_TOLERANCE_DEG) so
    coastal and border pipelines are not trimmed by boundary inaccuracies.
    """
    clip_area = canada_geometry.simplify(
        0.005, preserve_topology=True
    ).buffer(CLIP_TOLERANCE_DEG)

    # Prepared geometries make repeated intersects/contains tests much faster.
    prepared = prep(clip_area)

    def clip(geom):
        # Entirely outside Canada -> drop.
        if not prepared.intersects(geom):
            return None, "removed"

        # Entirely inside Canada -> keep untouched.
        if prepared.contains(geom):
            return geom, "kept"

        # Crosses the border -> trim to the Canadian portion.
        clipped = _keep_lines(geom.intersection(clip_area))
        if clipped is None:
            return None, "removed"

        return clipped, "clipped"

    return clip

### 3.3 GeoJSON conversion and outside-Canada mask

In [ ]:
def dataframe_to_geojson(df, canada_geometry):
    """Convert the cleaned dataframe to a GeoJSON FeatureCollection clipped to Canada."""

    clip_to_canada = build_canada_clipper(canada_geometry)

    features = []
    stats = {"kept": 0, "clipped": 0, "removed": 0}

    # Attributes copied into each feature (shown in popups and used by AI filters).
    columns = [
        "pipeline_name",
        "company",
        "parent_company",
        "province",
        "location_description",
        "from_location",
        "to_location",
        "commodity",
        "pipeline_length_km",
        "capacity",
        "status",
        "start_date",
        "end_date",
        "segment_number",
        "licence_number",
        "project_number",
        "authority_type",
        "data_source",
        "source_dataset",
        "source_record_id"
    ]

    for _, row in df.iterrows():

        geometry = row.get("geometry")

        if geometry is None or pd.isna(geometry):
            continue

        try:
            geom = wkt.loads(geometry) if isinstance(geometry, str) else geometry

            if geom is None or geom.is_empty:
                continue

        except Exception as e:
            print(f"Skipping invalid geometry: {e}")
            continue

        try:
            geom, result = clip_to_canada(geom)
        except Exception as e:
            print(f"Skipping geometry that could not be clipped: {e}")
            continue

        stats[result] += 1

        if geom is None:
            continue

        # Convert pandas/numpy values to plain JSON-safe types.
        properties = {}

        for col in columns:
            if col in df.columns:
                value = row[col]

                if pd.isna(value):
                    value = None
                elif hasattr(value, "item"):
                    value = value.item()
                elif not isinstance(value, (str, int, float, bool)):
                    value = str(value)

                properties[col] = value

        features.append({
            "type": "Feature",
            "geometry": mapping(geom),
            "properties": properties
        })

    print(
        f"Canada filter -> kept whole: {stats['kept']:,} | "
        f"trimmed at border: {stats['clipped']:,} | "
        f"removed (outside Canada): {stats['removed']:,}"
    )

    return {
        "type": "FeatureCollection",
        "features": features
    }


def create_outside_mask(canada_geometry):
    """Build a world-sized polygon with Canada cut out, used to grey out everything else."""

    world = box(-180, -85, 180, 85)
    outside_canada = world.difference(canada_geometry)

    return {
        "type": "Feature",
        "geometry": mapping(outside_canada),
        "properties": {"name": "Outside Canada"}
    }

### 3.4 HTML / JavaScript page template
The full web page: Mapbox map, pipeline layer with click popups, legend, and the chat sidebar. Values like `__PIPELINE_GEOJSON__` are placeholders filled in by `generate_pipeline_map()`. The chat sends the conversation to the notebook's `/chat` endpoint, asks the model for a JSON filter, and applies it to the map.

In [ ]:
MAP_HTML_TEMPLATE = r"""
<!DOCTYPE html>
<html lang="en">

<head>

    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">

    <title>Canadian Pipeline Network</title>

    <script src="https://api.mapbox.com/mapbox-gl-js/v3.9.4/mapbox-gl.js"></script>
    <link href="https://api.mapbox.com/mapbox-gl-js/v3.9.4/mapbox-gl.css"
          rel="stylesheet">

    <style>

        * { margin: 0; padding: 0; box-sizing: border-box; }

        html, body { height: 100%; }

        body {
            font-family: -apple-system, "Segoe UI", Roboto, Arial, sans-serif;
            overflow: hidden;
            background: #f4f5f7;
        }

        #app { display: flex; width: 100%; height: 100vh; }

        /* ---------------- SIDEBAR ---------------- */

        #sidebar {
            width: 360px;
            min-width: 360px;
            height: 100%;
            display: flex;
            flex-direction: column;
            background: #ffffff;
            border-right: 1px solid #e3e5e8;
            z-index: 20;
            transition: margin-left 0.25s ease;
        }

        #sidebar.collapsed { margin-left: -360px; }

        .sb-header {
            padding: 16px 18px;
            border-bottom: 1px solid #eceef1;
        }

        .sb-header h2 { font-size: 17px; color: #111; }

        .sb-header p { font-size: 12px; color: #6b7280; margin-top: 4px; }

        #filter-bar {
            display: none;
            align-items: center;
            justify-content: space-between;
            gap: 8px;
            padding: 8px 16px;
            background: #fff4f4;
            border-bottom: 1px solid #f5d0d0;
            font-size: 12px;
            color: #7a1f1f;
        }

        #filter-bar button {
            border: 1px solid #e04444;
            background: white;
            color: #e04444;
            border-radius: 6px;
            padding: 3px 9px;
            font-size: 12px;
            cursor: pointer;
            white-space: nowrap;
        }

        #messages {
            flex: 1;
            overflow-y: auto;
            padding: 16px;
            display: flex;
            flex-direction: column;
            gap: 10px;
            background: #fafbfc;
        }

        .msg {
            max-width: 88%;
            padding: 10px 13px;
            border-radius: 14px;
            font-size: 13.5px;
            line-height: 1.45;
            white-space: pre-wrap;
            word-wrap: break-word;
        }

        .msg.bot {
            align-self: flex-start;
            background: #eef0f3;
            color: #1f2937;
            border-bottom-left-radius: 4px;
        }

        .msg.user {
            align-self: flex-end;
            background: #e04444;
            color: #ffffff;
            border-bottom-right-radius: 4px;
        }

        .msg.typing { color: #6b7280; font-style: italic; }

        .msg.error { background: #fdecec; color: #8a1c1c; }

        #suggestions {
            display: flex;
            flex-wrap: wrap;
            gap: 6px;
            padding: 10px 16px 0 16px;
            background: #fafbfc;
        }

        .chip {
            border: 1px solid #d5d8dd;
            background: #ffffff;
            color: #374151;
            font-size: 12px;
            padding: 6px 10px;
            border-radius: 999px;
            cursor: pointer;
        }

        .chip:hover { background: #f1f2f4; }

        #chat-form {
            display: flex;
            gap: 8px;
            padding: 12px 16px 16px 16px;
            background: #fafbfc;
            border-top: 1px solid #eceef1;
        }

        #chat-input {
            flex: 1;
            resize: none;
            height: 42px;
            max-height: 120px;
            padding: 10px 12px;
            border: 1px solid #d5d8dd;
            border-radius: 10px;
            font-family: inherit;
            font-size: 13.5px;
            outline: none;
        }

        #chat-input:focus { border-color: #e04444; }

        #send-btn {
            border: none;
            background: #e04444;
            color: white;
            padding: 0 16px;
            border-radius: 10px;
            font-size: 13.5px;
            cursor: pointer;
        }

        #send-btn:hover { background: #c93a3a; }
        #send-btn:disabled { background: #e8a0a0; cursor: not-allowed; }

        /* ---------------- MAP AREA ---------------- */

        #map-wrap { position: relative; flex: 1; height: 100%; min-width: 0; }

        #map { width: 100%; height: 100%; }

        #toggle-btn {
            position: absolute;
            top: 12px;
            left: 12px;
            z-index: 15;
            width: 36px;
            height: 36px;
            border: none;
            border-radius: 8px;
            background: white;
            box-shadow: 0 2px 10px rgba(0,0,0,0.2);
            font-size: 16px;
            cursor: pointer;
        }

        .legend {
            position: absolute;
            bottom: 30px;
            left: 20px;
            z-index: 10;
            background: white;
            padding: 15px;
            border-radius: 8px;
            box-shadow: 0 2px 10px rgba(0,0,0,0.15);
            font-size: 13px;
        }

        .legend-item { display: flex; align-items: center; margin-top: 8px; }

        .legend-line { width: 25px; height: 4px; background: #e04444; margin-right: 10px; }

        .legend-mask { width: 25px; height: 14px; background: #999; margin-right: 10px; }

        /* ---------------- POPUP ---------------- */

        .mapboxgl-popup-content {
            padding: 15px;
            border-radius: 8px;
            min-width: 220px;
            max-width: 320px;
        }

        .popup-title { font-size: 16px; font-weight: bold; margin-bottom: 10px; }

        .popup-row { font-size: 12px; margin: 6px 0; color: #444; }

        .popup-row strong { color: #111; }

    </style>

</head>

<body>

    <div id="app">

        <aside id="sidebar">

            <div class="sb-header">
                <h2>Canadian Pipeline Network</h2>
                <p id="stats"></p>
            </div>

            <div id="filter-bar">
                <span id="filter-text"></span>
                <button id="clear-btn" type="button">Show all</button>
            </div>

            <div id="messages"></div>

            <div id="suggestions"></div>

            <form id="chat-form" autocomplete="off">
                <textarea id="chat-input"
                          placeholder="Ask about pipelines..."
                          rows="1"></textarea>
                <button id="send-btn" type="submit">Send</button>
            </form>

        </aside>

        <div id="map-wrap">

            <button id="toggle-btn" title="Show / hide chat">&#9776;</button>

            <div id="map"></div>

            <div class="legend">
                <strong>Map Legend</strong>
                <div class="legend-item"><div class="legend-line"></div>Pipeline Network</div>
                <div class="legend-item"><div class="legend-mask"></div>Outside Canada</div>
            </div>

        </div>

    </div>


    <script>

        // --------------------------------------------------
        // SETTINGS
        // --------------------------------------------------

        // Chat requests go to the /chat endpoint of the Python server
        // (which runs the local Qwen model), not to an external API.
        const CHAT_URL = "chat";


        // --------------------------------------------------
        // MAPBOX INITIALIZATION
        // --------------------------------------------------

        mapboxgl.accessToken = "__MAPBOX_TOKEN__";

        const map = new mapboxgl.Map({
            container: "map",
            style: "mapbox://styles/mapbox/light-v11",
            center: [__CENTER_LON__, __CENTER_LAT__],
            zoom: 3,
            minZoom: 2,
            maxZoom: 15,
            maxBounds: __CANADA_BOUNDS__,
            renderWorldCopies: false
        });

        map.addControl(new mapboxgl.NavigationControl(), "top-right");
        map.addControl(new mapboxgl.FullscreenControl(), "top-right");


        // --------------------------------------------------
        // DATA (pipelines already clipped to Canada)
        // --------------------------------------------------

        const pipelineData = __PIPELINE_GEOJSON__;
        const canadaData = __CANADA_GEOJSON__;
        const outsideMask = __OUTSIDE_MASK__;

        // Give every feature a unique id so the AI filter can select them
        // (the map filter uses these ids to show only matching pipelines)
        pipelineData.features.forEach(function(f, i) { f.properties._id = i; });


        // --------------------------------------------------
        // MAP LAYERS
        // --------------------------------------------------

        map.on("load", function() {

            map.addSource("canada", { type: "geojson", data: canadaData });
            map.addSource("outside-mask", { type: "geojson", data: outsideMask });

            map.addLayer({
                id: "outside-canada-fill",
                type: "fill",
                source: "outside-mask",
                paint: { "fill-color": "#999999", "fill-opacity": 0.78 }
            });

            map.addLayer({
                id: "canada-outline",
                type: "line",
                source: "canada",
                paint: {
                    "line-color": "#333333",
                    "line-width": 1.5,
                    "line-opacity": 0.9
                }
            });

            map.addSource("pipelines", { type: "geojson", data: pipelineData });

            map.addLayer({
                id: "pipeline-lines",
                type: "line",
                source: "pipelines",
                layout: { "line-join": "round", "line-cap": "round" },
                paint: {
                    "line-color": "#e04444",
                    "line-width": [
                        "interpolate", ["linear"], ["zoom"],
                        3, 1,
                        6, 2,
                        10, 3.5
                    ],
                    "line-opacity": 0.9
                }
            });

            function safe(value) {
                return String(value)
                    .replaceAll("&", "&amp;")
                    .replaceAll("<", "&lt;")
                    .replaceAll(">", "&gt;")
                    .replaceAll('"', "&quot;")
                    .replaceAll("'", "&#039;");
            }

            map.on("click", "pipeline-lines", function(e) {

                const p = e.features[0].properties;

                const popupHTML = `
                    <div class="popup-title">${safe(p.pipeline_name || "Unknown Pipeline")}</div>
                    <div class="popup-row"><strong>Company:</strong> ${safe(p.company || "Not available")}</div>
                    <div class="popup-row"><strong>Province:</strong> ${safe(p.province || "Not available")}</div>
                    <div class="popup-row"><strong>Commodity:</strong> ${safe(p.commodity || "Not available")}</div>
                    <div class="popup-row"><strong>Length:</strong> ${safe(p.pipeline_length_km ?? "Not available")} km</div>
                    <div class="popup-row"><strong>Capacity:</strong> ${safe(p.capacity || "Not available")}</div>
                    <div class="popup-row"><strong>Status:</strong> ${safe(p.status || "Not available")}</div>
                    <div class="popup-row"><strong>From:</strong> ${safe(p.from_location || "Not available")}</div>
                    <div class="popup-row"><strong>To:</strong> ${safe(p.to_location || "Not available")}</div>
                `;

                new mapboxgl.Popup({ closeButton: true, maxWidth: "320px" })
                    .setLngLat(e.lngLat)
                    .setHTML(popupHTML)
                    .addTo(map);
            });

            map.on("mouseenter", "pipeline-lines", function() {
                map.getCanvas().style.cursor = "pointer";
            });

            map.on("mouseleave", "pipeline-lines", function() {
                map.getCanvas().style.cursor = "";
            });

            fitCanada(1500);
        });


        function fitCanada(duration) {
            map.fitBounds(
                [[__MIN_LON__, __MIN_LAT__], [__MAX_LON__, __MAX_LAT__]],
                { padding: 40, duration: duration }
            );
        }


        // --------------------------------------------------
        // SIDEBAR: SHOW / HIDE
        // --------------------------------------------------

        const sidebar = document.getElementById("sidebar");

        document.getElementById("toggle-btn").addEventListener("click", function() {
            sidebar.classList.toggle("collapsed");
            setTimeout(function() { map.resize(); }, 300);
        });


        // --------------------------------------------------
        // AI: WHAT THE MODEL IS TOLD ABOUT YOUR DATA
        // --------------------------------------------------

        function topValues(field, limit) {
            const counts = {};
            pipelineData.features.forEach(function(f) {
                const v = f.properties[field];
                if (v !== null && v !== undefined && String(v).trim() !== "") {
                    counts[v] = (counts[v] || 0) + 1;
                }
            });
            return Object.keys(counts)
                .sort(function(a, b) { return counts[b] - counts[a]; })
                .slice(0, limit);
        }

        const KNOWN_PROVINCES = topValues("province", 30);
        const KNOWN_COMMODITIES = topValues("commodity", 40);
        const KNOWN_STATUSES = topValues("status", 20);

        const SYSTEM_PROMPT = [
            "You are the assistant inside a map of Canadian pipelines.",
            "Convert the user's request into a JSON filter for the map. Reply with JSON ONLY, no markdown, no extra text.",
            "",
            "JSON format:",
            '{"action": "filter" | "reset" | "chat",',
            ' "filters": {',
            '   "provinces": [strings],',
            '   "commodities": [strings],',
            '   "statuses": [strings],',
            '   "companies": [strings],',
            '   "name_contains": string or null,',
            '   "min_length_km": number or null,',
            '   "max_length_km": number or null,',
            '   "top_longest": integer or null',
            ' },',
            ' "reply": "one short friendly sentence"}',
            "",
            "Rules:",
            '- action "filter": show only pipelines matching the filters. Leave unused filter fields as [] or null.',
            '- action "reset": the user wants to see all pipelines again.',
            '- action "chat": the message is not a map request (greeting, question about what you can do). Put your answer in reply.',
            "- You will be given the CURRENT filters. If the user is refining (e.g. 'only the ones in Alberta', 'now just gas'), return the full merged filters. If it is a new topic, start fresh.",
            "- provinces, commodities and statuses: use values from the lists below, copied exactly. Pick every value that fits (e.g. 'oil' may match several commodity values).",
            "- companies: partial company names, matched against company and parent company.",
            "- top_longest: use when the user asks for the longest N pipelines.",
            "- Never invent data. You only control the map filters.",
            "",
            "Provinces in the data: " + JSON.stringify(KNOWN_PROVINCES),
            "Commodities in the data: " + JSON.stringify(KNOWN_COMMODITIES),
            "Statuses in the data: " + JSON.stringify(KNOWN_STATUSES)
        ].join("\n");


        // --------------------------------------------------
        // AI: APPLY A FILTER TO THE MAP
        // --------------------------------------------------

        let currentFilters = null;

        function asStringList(value) {
            if (!Array.isArray(value)) return [];
            return value
                .map(function(v) { return String(v).trim(); })
                .filter(function(v) { return v.length > 0; });
        }

        function asNumberOrNull(value) {
            if (value === null || value === undefined || value === "") return null;
            const n = Number(value);
            return Number.isFinite(n) ? n : null;
        }

        function cleanFilters(raw) {
            raw = raw || {};
            return {
                provinces: asStringList(raw.provinces),
                commodities: asStringList(raw.commodities),
                statuses: asStringList(raw.statuses),
                companies: asStringList(raw.companies),
                name_contains: raw.name_contains ? String(raw.name_contains).trim() : null,
                min_length_km: asNumberOrNull(raw.min_length_km),
                max_length_km: asNumberOrNull(raw.max_length_km),
                top_longest: asNumberOrNull(raw.top_longest)
            };
        }

        function filtersAreEmpty(f) {
            return !f.provinces.length && !f.commodities.length &&
                   !f.statuses.length && !f.companies.length &&
                   !f.name_contains && f.min_length_km === null &&
                   f.max_length_km === null && f.top_longest === null;
        }

        function textMatches(value, terms) {
            if (!terms.length) return true;
            const v = String(value === null || value === undefined ? "" : value).toLowerCase();
            if (!v) return false;
            return terms.some(function(t) { return v.includes(t.toLowerCase()); });
        }

        function findMatches(f) {

            let matches = pipelineData.features.filter(function(feature) {

                const p = feature.properties;

                if (!textMatches(p.province, f.provinces)) return false;
                if (!textMatches(p.commodity, f.commodities)) return false;
                if (!textMatches(p.status, f.statuses)) return false;

                if (f.companies.length) {
                    const inCompany = textMatches(p.company, f.companies);
                    const inParent = textMatches(p.parent_company, f.companies);
                    if (!inCompany && !inParent) return false;
                }

                if (f.name_contains) {
                    if (!textMatches(p.pipeline_name, [f.name_contains])) return false;
                }

                const len = (typeof p.pipeline_length_km === "number") ? p.pipeline_length_km : null;

                if (f.min_length_km !== null && (len === null || len < f.min_length_km)) return false;
                if (f.max_length_km !== null && (len === null || len > f.max_length_km)) return false;

                return true;
            });

            if (f.top_longest !== null && f.top_longest > 0) {
                matches = matches.slice().sort(function(a, b) {
                    const la = (typeof a.properties.pipeline_length_km === "number") ? a.properties.pipeline_length_km : -1;
                    const lb = (typeof b.properties.pipeline_length_km === "number") ? b.properties.pipeline_length_km : -1;
                    return lb - la;
                }).slice(0, Math.floor(f.top_longest));
            }

            return matches;
        }

        function extendBounds(b, coords) {
            if (typeof coords[0] === "number") {
                b[0] = Math.min(b[0], coords[0]);
                b[1] = Math.min(b[1], coords[1]);
                b[2] = Math.max(b[2], coords[0]);
                b[3] = Math.max(b[3], coords[1]);
            } else {
                coords.forEach(function(c) { extendBounds(b, c); });
            }
        }

        function zoomToFeatures(features) {
            const b = [Infinity, Infinity, -Infinity, -Infinity];
            features.forEach(function(f) { extendBounds(b, f.geometry.coordinates); });
            if (!Number.isFinite(b[0])) return;
            map.fitBounds([[b[0], b[1]], [b[2], b[3]]], {
                padding: 60,
                duration: 1200,
                maxZoom: 9
            });
        }

        function describeFilters(f) {
            const parts = [];
            if (f.provinces.length) parts.push(f.provinces.join(", "));
            if (f.commodities.length) parts.push(f.commodities.join(", "));
            if (f.statuses.length) parts.push(f.statuses.join(", "));
            if (f.companies.length) parts.push(f.companies.join(", "));
            if (f.name_contains) parts.push('name: "' + f.name_contains + '"');
            if (f.min_length_km !== null) parts.push("min " + f.min_length_km + " km");
            if (f.max_length_km !== null) parts.push("max " + f.max_length_km + " km");
            if (f.top_longest !== null) parts.push("top " + f.top_longest + " longest");
            return parts.join(" | ");
        }

        function summarize(matches) {
            let totalKm = 0;
            const companyKm = {};

            matches.forEach(function(feature) {
                const p = feature.properties;
                if (typeof p.pipeline_length_km === "number") {
                    totalKm += p.pipeline_length_km;
                    const c = p.company || "Unknown";
                    companyKm[c] = (companyKm[c] || 0) + p.pipeline_length_km;
                }
            });

            let text = matches.length.toLocaleString() + " pipeline segments shown";

            if (totalKm > 0) {
                text += ", about " + Math.round(totalKm).toLocaleString() + " km in total";
            }

            const topCompany = Object.keys(companyKm).sort(function(a, b) {
                return companyKm[b] - companyKm[a];
            })[0];

            if (topCompany) {
                text += ". Largest operator in this view: " + topCompany + ".";
            }

            return text;
        }

        function showAllPipelines() {
            currentFilters = null;

            if (map.getLayer("pipeline-lines")) {
                map.setFilter("pipeline-lines", null);
            }

            document.getElementById("filter-bar").style.display = "none";
            fitCanada(1000);
        }

        // Returns a status line, or null if nothing matched (map left untouched)
        function applyFilters(filters) {

            if (filtersAreEmpty(filters)) {
                showAllPipelines();
                return pipelineData.features.length.toLocaleString() + " pipeline segments shown (no filters).";
            }

            const matches = findMatches(filters);

            if (!matches.length) return null;

            const ids = matches.map(function(f) { return f.properties._id; });

            map.setFilter("pipeline-lines", ["in", ["get", "_id"], ["literal", ids]]);

            currentFilters = filters;

            document.getElementById("filter-text").textContent = "Filter: " + describeFilters(filters);
            document.getElementById("filter-bar").style.display = "flex";

            zoomToFeatures(matches);

            return summarize(matches);
        }

        document.getElementById("clear-btn").addEventListener("click", function() {
            showAllPipelines();
            addMessage("Filters cleared. Showing all pipelines.", "bot");
        });


        // --------------------------------------------------
        // AI: CALL THE LOCAL MODEL
        // --------------------------------------------------

        const history = [];

        async function askModel(userText) {

            const messages = [{ role: "system", content: SYSTEM_PROMPT }]
                .concat(history.slice(-6))
                .concat([{
                    role: "user",
                    content:
                        "CURRENT filters: " + JSON.stringify(currentFilters) +
                        "\nUser request: " + userText
                }]);

            const response = await fetch(CHAT_URL, {
                method: "POST",
                headers: { "Content-Type": "application/json" },
                body: JSON.stringify({
                    messages: messages,
                    temperature: 0.1,
                    max_tokens: 500
                })
            });

            if (!response.ok) {
                const detail = await response.text();
                throw new Error("Chat server returned " + response.status + ". " + detail.slice(0, 200));
            }

            const data = await response.json();
            const raw = data.choices[0].message.content || "";

            history.push({ role: "user", content: userText });
            history.push({ role: "assistant", content: raw });

            return raw;
        }

        function parseModelJson(raw) {
            const start = raw.indexOf("{");
            const end = raw.lastIndexOf("}");
            if (start === -1 || end <= start) return null;
            try {
                return JSON.parse(raw.slice(start, end + 1));
            } catch (err) {
                return null;
            }
        }

        async function getBotReply(userText) {

            const raw = await askModel(userText);
            const result = parseModelJson(raw);

            if (!result) {
                return {
                    text: "I couldn't turn that into a map filter. Try something like " +
                          '"show oil pipelines in Alberta".',
                    isError: true
                };
            }

            const reply = result.reply ? String(result.reply) : "";

            if (result.action === "reset") {
                showAllPipelines();
                return { text: reply || "Showing all pipelines again." };
            }

            if (result.action === "filter") {
                const filters = cleanFilters(result.filters);
                const status = applyFilters(filters);

                if (status === null) {
                    return {
                        text: "No pipelines matched that search, so I left the map as it was. " +
                              "Try broader terms.",
                        isError: true
                    };
                }

                return { text: (reply ? reply + "\n\n" : "") + status };
            }

            return { text: reply || "I can filter the map by province, commodity, company, status and length." };
        }


        // --------------------------------------------------
        // SIDEBAR: CHAT UI
        // --------------------------------------------------

        document.getElementById("stats").textContent =
            pipelineData.features.length.toLocaleString() +
            " pipeline segments (Canada only)";

        const messagesEl = document.getElementById("messages");
        const inputEl = document.getElementById("chat-input");
        const formEl = document.getElementById("chat-form");
        const sendBtn = document.getElementById("send-btn");
        const suggestionsEl = document.getElementById("suggestions");

        function addMessage(text, role, extraClass) {
            const div = document.createElement("div");
            div.className = "msg " + role + (extraClass ? " " + extraClass : "");
            div.textContent = text;
            messagesEl.appendChild(div);
            messagesEl.scrollTop = messagesEl.scrollHeight;
            return div;
        }

        async function handleUserMessage(text) {

            text = text.trim();
            if (!text) return;

            addMessage(text, "user");

            const typing = addMessage("Thinking...", "bot", "typing");
            sendBtn.disabled = true;

            try {
                const result = await getBotReply(text);
                typing.remove();
                addMessage(result.text, "bot", result.isError ? "error" : "");
            } catch (err) {
                typing.remove();
                addMessage("Something went wrong: " + err.message, "bot", "error");
            } finally {
                sendBtn.disabled = false;
                inputEl.focus();
            }
        }

        formEl.addEventListener("submit", function(e) {
            e.preventDefault();
            const text = inputEl.value;
            inputEl.value = "";
            handleUserMessage(text);
        });

        inputEl.addEventListener("keydown", function(e) {
            if (e.key === "Enter" && !e.shiftKey) {
                e.preventDefault();
                formEl.requestSubmit();
            }
        });

        [
            "Show oil pipelines in Alberta",
            "Show the 5 longest pipelines",
            "Pipelines longer than 500 km"
        ].forEach(function(label) {
            const chip = document.createElement("button");
            chip.type = "button";
            chip.className = "chip";
            chip.textContent = label;
            chip.addEventListener("click", function() { handleUserMessage(label); });
            suggestionsEl.appendChild(chip);
        });

        addMessage(
            "Hi! Ask me to filter the map in plain English, for example " +
            '"natural gas pipelines in British Columbia" or "only the ones longer than 300 km". ' +
            "Say \"show all\" to reset.",
            "bot"
        );

    </script>

</body>

</html>
"""

### 3.5 Page generator

In [ ]:
def generate_pipeline_map(pipelines_df):
    """
    Build the complete map page (HTML string) from the cleaned pipeline dataframe.

    Steps: download the Canada boundary, clip pipelines to it, build the grey
    "outside Canada" mask, then inject all data and settings into MAP_HTML_TEMPLATE.
    """

    canada_geometry = get_canada_boundary()

    print("Preparing pipeline geometries (Canada only)...")

    pipeline_geojson = dataframe_to_geojson(pipelines_df, canada_geometry)

    print(
        f"Successfully converted "
        f"{len(pipeline_geojson['features'])} pipeline features."
    )

    canada_geojson = {
        "type": "Feature",
        "geometry": mapping(canada_geometry),
        "properties": {"name": "Canada"}
    }

    outside_mask = create_outside_mask(canada_geometry)

    # Map centre = middle of Canada's bounding box
    min_lon, min_lat, max_lon, max_lat = canada_geometry.bounds
    center_lon = (min_lon + max_lon) / 2
    center_lat = (min_lat + max_lat) / 2

    # Fill the __PLACEHOLDERS__ in the template with real values
    replacements = {
        "__MAPBOX_TOKEN__": html.escape(MAPBOX_TOKEN, quote=True),
        "__CENTER_LON__": str(center_lon),
        "__CENTER_LAT__": str(center_lat),
        "__CANADA_BOUNDS__": json.dumps(CANADA_BOUNDS),
        "__PIPELINE_GEOJSON__": json.dumps(pipeline_geojson),
        "__CANADA_GEOJSON__": json.dumps(canada_geojson),
        "__OUTSIDE_MASK__": json.dumps(outside_mask),
        "__MIN_LON__": str(min_lon),
        "__MIN_LAT__": str(min_lat),
        "__MAX_LON__": str(max_lon),
        "__MAX_LAT__": str(max_lat)
    }

    html_content = MAP_HTML_TEMPLATE
    for key, value in replacements.items():
        html_content = html_content.replace(key, value)

    print(f"Pipeline features (Canada only): {len(pipeline_geojson['features'])}")

    return html_content

## 4. Local AI model
Loads an open **Qwen2.5 Instruct** model (no login required): **3B in float16 on a GPU** (T4), or **1.5B on CPU** as a fallback. The first run downloads the weights (1–3 minutes).

### 4.1 Load the model

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

USE_GPU = torch.cuda.is_available()

# Larger model on GPU, smaller one on CPU
LOCAL_MODEL = "Qwen/Qwen2.5-3B-Instruct" if USE_GPU else "Qwen/Qwen2.5-1.5B-Instruct"
print(f"GPU available: {USE_GPU} | loading {LOCAL_MODEL} (first run downloads it, 1-3 min)...")

tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL)
dtype = torch.float16 if USE_GPU else torch.float32

try:
    model = AutoModelForCausalLM.from_pretrained(LOCAL_MODEL, dtype=dtype, low_cpu_mem_usage=True)
except TypeError:
    # Older transformers versions use the old argument name
    model = AutoModelForCausalLM.from_pretrained(LOCAL_MODEL, torch_dtype=dtype, low_cpu_mem_usage=True)

if USE_GPU:
    model = model.to("cuda")

model.eval()
print("Model ready.")

### 4.2 Chat function
Runs one chat completion. The answer is pre-filled with `{` so the model has to continue with JSON, which the map page then parses into filters. Decoding is greedy (`do_sample=False`) for consistent results.

In [ ]:
def run_chat(messages, max_new_tokens=300):
    """Run the local model on a list of {role, content} messages and return its JSON reply."""
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    # Start the answer with "{" so the model is forced to reply with JSON
    prompt += "{"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    # Decode only the newly generated tokens and re-attach the leading "{"
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return "{" + tokenizer.decode(new_tokens, skip_special_tokens=True)

## 5. Run

### 5.1 Clean the datasets
Produces `pipelines_df` and saves `/content/data/Canadian_Pipelines_Clean.csv`.

In [ ]:
pipelines_df = clean_pipeline_dataset(file_paths)

print("Cleaned dataset shape:", pipelines_df.shape)
display(pipelines_df.head())

### 5.2 Launch the map
Builds the map page and starts a small web server inside the notebook:

- `GET /` serves the map page
- `POST /chat` runs the local model for the chat sidebar

Click the link printed below to open the map in a new browser tab. Keep the runtime running while you use it.

In [ ]:
PORT = 8765

# Build the map page
page = generate_pipeline_map(pipelines_df)

# Shared state for the request handler (re-running this cell just swaps the page)
PAGE = {"html": page}
generation_lock = threading.Lock()  # the model answers one question at a time


class Handler(BaseHTTPRequestHandler):
    """Serves the map page and answers chat requests."""

    def log_message(self, *args):
        pass  # keep the notebook output clean

    def _send(self, status, body, content_type):
        data = body.encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", content_type)
        self.send_header("Content-Length", str(len(data)))
        self.end_headers()
        self.wfile.write(data)

    def do_GET(self):
        # Map page
        if self.path.split("?")[0] in ("/", "/index.html"):
            self._send(200, PAGE["html"], "text/html; charset=utf-8")
        else:
            self._send(404, "Not found", "text/plain")

    def do_POST(self):
        # Chat endpoint used by the sidebar
        if self.path.split("?")[0] != "/chat":
            self._send(404, "Not found", "text/plain")
            return

        try:
            length = int(self.headers.get("Content-Length", 0))
            payload = json.loads(self.rfile.read(length))

            with generation_lock:
                content = run_chat(payload["messages"])

            # Same response shape as an OpenAI-style chat API, which the page expects
            result = {"choices": [{"message": {"content": content}}]}
            self._send(200, json.dumps(result), "application/json")

        except Exception as error:
            self._send(500, json.dumps({"error": str(error)}), "application/json")


# Start the server only once; re-running the cell just updates the page
if "httpd" not in globals():
    httpd = ThreadingHTTPServer(("0.0.0.0", PORT), Handler)
    threading.Thread(target=httpd.serve_forever, daemon=True).start()
    print(f"Server started on port {PORT}.")
else:
    print("Server already running. Page updated.")

# Expose the port through Colab and print a link to open it in a new tab
print("\nClick the link below to open the map in a new tab:")
output.serve_kernel_port_as_window(PORT)

## Troubleshooting

- **Dataset not found:** update `DATA_FOLDER` / `file_paths` in section 1.4.
- **Mapbox token error or blank map:** check the `MAPBOX_ACCESS_TOKEN` secret, make sure notebook access is enabled, and use a valid public (`pk.`) token.
- **Chat is slow or the model runs on CPU:** set **Runtime → Change runtime type → T4 GPU**, then restart and run all cells.
- **Chat says it could not turn the request into a filter:** rephrase, e.g. *"show natural gas pipelines in British Columbia"*.
- **Map link stops working:** the Colab runtime was disconnected; re-run section 5.
- **Changed the data or code:** re-run 5.1 and 5.2 (the server keeps running and just serves the new page).